In [1]:
import os
import gc
import glob
import hashlib

import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# ---------------- ตั้งค่า ----------------
DATASET_ROOT = os.environ.get("DATASET_ROOT", "/home/jovyan/shared/donut/Alzheimer/OAS1")
SEED = 1334
CHECK_ID = "0454"        # OASIS ใช้ ID 4 หลัก เปลี่ยนเป็นคนอื่นได้ (ต้องมี CDR และมีไฟล์ภาพ)
N_RUNS = 5

SIZE_2D = 224            # 2D และ 2.5D ใช้ 224 x 224
SHAPE_3D = (128, 128, 96)
MODES = ["2D", "2.5D", "3D"]
EXPECTED = {"2D": (1, SIZE_2D, SIZE_2D), "2.5D": (3, SIZE_2D, SIZE_2D), "3D": (1,) + SHAPE_3D}
# -----------------------------------------


In [2]:
df = pd.read_excel(f"{DATASET_ROOT}/_oasis_cross-sectional.xlsx")
df = df[df["CDR"].notna()]
df["label"] = (df["CDR"] > 0).astype(int)

def find_file(subject_id):
    pattern = f"{DATASET_ROOT}/{subject_id}/PROCESSED/MPRAGE/T88_111/*_masked_gfc.hdr"
    files = glob.glob(pattern)
    if len(files) == 0:
        return None
    return files[0]

df["path"] = df["ID"].apply(find_file)
df = df[df["path"].notna()]
df["subject_id"] = df["ID"].apply(lambda x: x.split("_")[1])

print("df หลังกรอง:", df.shape)
print("จำนวน subject ไม่ซ้ำ:", df["subject_id"].nunique())
print(f"{CHECK_ID} อยู่ใน df:", CHECK_ID in set(df["subject_id"]))


df หลังกรอง: (234, 15)
จำนวน subject ไม่ซ้ำ: 234
0454 อยู่ใน df: True


In [3]:
def make_split(df, seed):
    # เหมือนเซลล์แบ่งข้อมูลใน Resnet18.ipynb ทุกประการ
    subject_labels = df.groupby("subject_id")["label"].first()
    subject_ids = list(subject_labels.index)
    subject_vals = list(subject_labels.values)

    train_subjects, temp_subjects = train_test_split(
        subject_ids, test_size=0.3, stratify=subject_vals, random_state=seed
    )

    temp_idx_map = dict(zip(subject_ids, subject_vals))
    temp_labels_list = [temp_idx_map[s] for s in temp_subjects]

    val_subjects, test_subjects = train_test_split(
        temp_subjects, test_size=0.5, stratify=temp_labels_list, random_state=seed
    )

    train_df = df[df["subject_id"].isin(train_subjects)]
    val_df = df[df["subject_id"].isin(val_subjects)]
    test_df = df[df["subject_id"].isin(test_subjects)]

    # แก้ assert เดิม: เช็คทีละคู่ (ของเดิมตัดกันสามชุดพร้อมกัน ซึ่งว่างเสมอ)
    tr, va, te = set(train_subjects), set(val_subjects), set(test_subjects)
    assert not (tr & va) and not (tr & te) and not (va & te)

    return train_df, val_df, test_df


In [4]:
def load_and_prep(path):
    # ส่วนที่ทุกโมเดลใช้ร่วมกัน: โหลด -> crop สมอง -> normalize (เหมือนของเดิม ยังไม่ resize)
    img = nib.load(path)
    volume = img.get_fdata()

    if volume.ndim == 4:
        volume = volume[:, :, :, 0]

    nz = np.argwhere(volume > 0)
    lo, hi = nz.min(axis=0), nz.max(axis=0)
    volume = volume[lo[0]:hi[0]+1, lo[1]:hi[1]+1, lo[2]:hi[2]+1]

    p_lo, p_hi = np.percentile(volume, [1, 99])
    volume = np.clip(volume, p_lo, p_hi)
    volume = (volume - p_lo) / (p_hi - p_lo + 1e-8)
    return volume


def resize2d(s):
    return zoom(s, (SIZE_2D / s.shape[0], SIZE_2D / s.shape[1]), order=1)


def to_target(volume, mode):
    # *** ถ้า 2D / 2.5D ของคุณตัดภาพต่างจากนี้ ให้แก้ตรงนี้ ***
    if mode == "3D":
        factor = [t / c for t, c in zip(SHAPE_3D, volume.shape)]
        return zoom(volume, factor, order=1)               # (128, 128, 96)

    k = volume.shape[2] // 2                                # slice กลางแกน z
    if mode == "2D":
        return resize2d(volume[:, :, k])                    # (224, 224)
    if mode == "2.5D":
        return np.stack([resize2d(volume[:, :, k + d]) for d in (-1, 0, 1)])   # (3, 224, 224)
    raise ValueError(mode)


def preload(frame, mode, desc):
    # preload เฉพาะโหมดที่ระบุ (ไม่ทำพร้อมกัน)
    imgs, labels, sids = [], [], []
    for path, lb, sid in tqdm(
        zip(frame["path"].tolist(), frame["label"].tolist(), frame["subject_id"].tolist()),
        total=len(frame), desc=desc, leave=False
    ):
        imgs.append(to_target(load_and_prep(path), mode))
        labels.append(lb)
        sids.append(sid)
    x = np.stack(imgs).astype(np.float32)
    if mode in ("2D", "3D"):
        x = x[:, None]                                      # เพิ่ม channel = 1
    return x, np.array(labels), sids


def checksum(arr):
    return hashlib.sha1(np.ascontiguousarray(arr).tobytes()).hexdigest()[:10]


In [5]:
records = []

for mode in MODES:                      # ทำทีละโมเดล: 2D ครบ 5 รอบ -> 2.5D ครบ 5 รอบ -> 3D ครบ 5 รอบ
    print("#" * 78)
    print(f"โมเดล {mode}  target shape = {EXPECTED[mode]}")

    for run in range(1, N_RUNS + 1):
        # ขั้นที่ 1: แบ่ง subject (เริ่มใหม่ทุกรอบ)
        train_df, val_df, test_df = make_split(df, SEED)

        # ขั้นที่ 2: preload ภาพของโหมดนี้เท่านั้น
        x_tr, _, s_tr = preload(train_df, mode, f"{mode} รอบ {run} train")
        x_va, _, s_va = preload(val_df, mode, f"{mode} รอบ {run} val")
        x_te, _, s_te = preload(test_df, mode, f"{mode} รอบ {run} test")

        for nm, x in [("train", x_tr), ("val", x_va), ("test", x_te)]:
            assert x.shape[1:] == EXPECTED[mode], f"{mode} {nm} shape ผิด: {x.shape}"

        # ขั้นที่ 3: หา CHECK_ID ในข้อมูลที่ preload แล้ว
        loaded = {"train": (s_tr, x_tr), "val": (s_va, x_va), "test": (s_te, x_te)}
        hits = [(nm, sids.index(CHECK_ID), checksum(x[sids.index(CHECK_ID)]))
                for nm, (sids, x) in loaded.items() if CHECK_ID in sids]

        if len(hits) == 0:
            where, idx, ck = "ไม่พบ", None, None
        else:
            assert len(hits) == 1, f"{CHECK_ID} ปรากฏมากกว่า 1 ชุด: {hits}"
            where, idx, ck = hits[0]

        a, b, c = set(s_tr), set(s_va), set(s_te)
        overlap = len(a & b) + len(a & c) + len(b & c)
        split_hash = hashlib.sha1(
            "|".join([",".join(sorted(a)), ",".join(sorted(b)), ",".join(sorted(c))]).encode()
        ).hexdigest()[:10]

        loc = f"{CHECK_ID} -> {where}" + (f" (ตำแหน่ง {idx}, checksum {ck})" if idx is not None else " (ไม่มี CDR หรือไม่มีไฟล์ภาพ จึงไม่เข้า df)")
        print(f"  {mode:5s} รอบ {run}: train={len(s_tr)} val={len(s_va)} test={len(s_te)} x_train{tuple(x_tr.shape)} | ซ้ำข้ามชุด={overlap} | {loc}")

        records.append({"mode": mode, "run": run, "where": where, "index": idx,
                        "image_checksum": ck, "n_train": len(s_tr), "n_val": len(s_va),
                        "n_test": len(s_te), "split_hash": split_hash})

        del x_tr, x_va, x_te, loaded
        gc.collect()


##############################################################################
โมเดล 2D  target shape = (1, 224, 224)


  2D    รอบ 1: train=163 val=35 test=36 x_train(163, 1, 224, 224) | ซ้ำข้ามชุด=0 | 0454 -> val (ตำแหน่ง 34, checksum d8689fbdcd)


  2D    รอบ 2: train=163 val=35 test=36 x_train(163, 1, 224, 224) | ซ้ำข้ามชุด=0 | 0454 -> val (ตำแหน่ง 34, checksum d8689fbdcd)


  2D    รอบ 3: train=163 val=35 test=36 x_train(163, 1, 224, 224) | ซ้ำข้ามชุด=0 | 0454 -> val (ตำแหน่ง 34, checksum d8689fbdcd)


  2D    รอบ 4: train=163 val=35 test=36 x_train(163, 1, 224, 224) | ซ้ำข้ามชุด=0 | 0454 -> val (ตำแหน่ง 34, checksum d8689fbdcd)


  2D    รอบ 5: train=163 val=35 test=36 x_train(163, 1, 224, 224) | ซ้ำข้ามชุด=0 | 0454 -> val (ตำแหน่ง 34, checksum d8689fbdcd)
##############################################################################
โมเดล 2.5D  target shape = (3, 224, 224)


... (6 outputs left out)

  3D    รอบ 2: train=163 val=35 test=36 x_train(163, 1, 128, 128, 96) | ซ้ำข้ามชุด=0 | 0454 -> val (ตำแหน่ง 34, checksum 06f99be104)


  3D    รอบ 3: train=163 val=35 test=36 x_train(163, 1, 128, 128, 96) | ซ้ำข้ามชุด=0 | 0454 -> val (ตำแหน่ง 34, checksum 06f99be104)


  3D    รอบ 4: train=163 val=35 test=36 x_train(163, 1, 128, 128, 96) | ซ้ำข้ามชุด=0 | 0454 -> val (ตำแหน่ง 34, checksum 06f99be104)


  3D    รอบ 5: train=163 val=35 test=36 x_train(163, 1, 128, 128, 96) | ซ้ำข้ามชุด=0 | 0454 -> val (ตำแหน่ง 34, checksum 06f99be104)


In [6]:
summary = pd.DataFrame(records)
print(summary.to_string(index=False))
print()

def same(series):
    # True/False ถ้ามีค่าให้เทียบ, "ไม่มีข้อมูล" ถ้า CHECK_ID ไม่อยู่ใน df (ไม่มีอะไรให้เทียบ)
    s = series.dropna().astype(str)
    return "ไม่มีข้อมูล" if s.empty else s.nunique() == 1

print("ภายในแต่ละโมเดล (5 รอบ):")
for mode, g in summary.groupby("mode", sort=False):
    pos = sorted(g["index"].dropna().astype(int).astype(str).unique())
    print(f"  {mode:5s}: {CHECK_ID} -> {', '.join(g['where'].unique())}"
          f" | ตำแหน่ง {pos if pos else '-'}"
          f" | ภาพเดิมทุกรอบ: {same(g['image_checksum'])}"
          f" | รายชื่อ 3 ชุดเหมือนกันทุกรอบ: {g['split_hash'].nunique() == 1}")

print()
print("ข้ามโมเดล (2D / 2.5D / 3D):")
print(f"  {CHECK_ID} อยู่ชุดเดียวกันทั้ง 3 โมเดล : {summary['where'].nunique() == 1}  ({', '.join(summary['where'].unique())})")
print(f"  ตำแหน่งในชุดตรงกันทั้ง 3 โมเดล        : {same(summary['index'])}")
print(f"  รายชื่อ train/val/test เหมือนกันทั้ง 3 โมเดล : {summary['split_hash'].nunique() == 1}")


mode  run where  index image_checksum  n_train  n_val  n_test split_hash
  2D    1   val     34     d8689fbdcd      163     35      36 67c0d18d36
  2D    2   val     34     d8689fbdcd      163     35      36 67c0d18d36
  2D    3   val     34     d8689fbdcd      163     35      36 67c0d18d36
  2D    4   val     34     d8689fbdcd      163     35      36 67c0d18d36
  2D    5   val     34     d8689fbdcd      163     35      36 67c0d18d36
2.5D    1   val     34     af289a12dd      163     35      36 67c0d18d36
2.5D    2   val     34     af289a12dd      163     35      36 67c0d18d36
2.5D    3   val     34     af289a12dd      163     35      36 67c0d18d36
2.5D    4   val     34     af289a12dd      163     35      36 67c0d18d36
2.5D    5   val     34     af289a12dd      163     35      36 67c0d18d36
  3D    1   val     34     06f99be104      163     35      36 67c0d18d36
  3D    2   val     34     06f99be104      163     35      36 67c0d18d36
  3D    3   val     34     06f99be104      163     